# Solution: 2026 CV competition

Object detection на основе **YOLO11l**.

**Воспроизводимость**: ноутбук запускается кнопкой *Run All* сверху вниз. 
Все сиды (Python / NumPy / PyTorch / cuDNN / Ultralytics) зафиксированы.

**Данные**: ожидается архив `2026-cv-competition.zip` в той же папке, что и этот ноутбук. 
Он будет распакован в `./workdir/competition_data`.

**Pretrained-модель**: `yolo11l.pt` подгружается из интернета пакетом `ultralytics` 
при первом запуске обучения, веса в репозитории не хранятся.

**Результат**: файл `submission.csv` в корне репозитория.

## 0. Установка зависимостей

In [ ]:
# Ставим ultralytics (тянет за собой совместимый torch/torchvision, opencv, pyyaml, tqdm и т.д.)
%pip install -q ultralytics scikit-learn

## 1. Импорты и фиксация сидов

Один общий `SEED` пробрасывается во все источники недетерминизма.

In [ ]:
import os
import gc
import csv
import time
import random
import shutil
import zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import yaml
import cv2
from PIL import Image
import torch

SEED = 993

def seed_everything(seed: int = SEED) -> None:
    """Фиксируем все источники случайности для воспроизводимости."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("SEED:", SEED)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Распаковка датасета

Архив `2026-cv-competition.zip` ожидается **в той же папке, что и этот ноутбук** 
(т.е. в корне репозитория). Распаковываем в `./workdir/competition_data`.

Внутри архива структура такая:
```
2026-cv-competition/
├── sample_submission.csv
├── train/train/{images,labels}
└── test/test/images
```

На случай, если внутри окажется чуть другой layout (бывает после разных архиваторов), 
пути к `train/test/sample_submission` ищем автопоиском, а не хардкодом.

In [ ]:
# Корнем считаем папку, где лежит ноутбук (а не cwd, который у Jupyter может отличаться).
REPO_ROOT = Path.cwd().resolve()
ARCHIVE_PATH = REPO_ROOT / "2026-cv-competition.zip"

WORKDIR = REPO_ROOT / "workdir"
DATA_DIR = WORKDIR / "competition_data"
WORKDIR.mkdir(parents=True, exist_ok=True)

assert ARCHIVE_PATH.exists(), (
    f"Не найден архив с данными: {ARCHIVE_PATH}\n"
    f"Положите 2026-cv-competition.zip в корень репозитория рядом с solution.ipynb."
)

FORCE_UNZIP = True  # перепаковываем при каждом прогоне — гарантирует чистое состояние

if FORCE_UNZIP and DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Распаковываем {ARCHIVE_PATH.name} -> {DATA_DIR}")
with zipfile.ZipFile(ARCHIVE_PATH, "r") as z:
    z.extractall(DATA_DIR)

# Чистим мусор от macOS-архиваторов
for junk_dir in DATA_DIR.rglob("__MACOSX"):
    if junk_dir.is_dir():
        shutil.rmtree(junk_dir)
for junk_file in DATA_DIR.rglob(".DS_Store"):
    junk_file.unlink(missing_ok=True)

print("Готово. Верхний уровень распакованного архива:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ", p.relative_to(DATA_DIR))

In [ ]:
# --- Утилиты для поиска файлов ---
def is_junk_path(p: Path) -> bool:
    s = str(p)
    return "__MACOSX" in s or p.name.startswith("._") or p.name == ".DS_Store"

def list_images(folder: Path):
    if not folder.exists():
        return []
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMG_EXTS and not is_junk_path(p)
    )

def list_labels(folder: Path):
    if not folder.exists():
        return []
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() == ".txt" and not is_junk_path(p)
    )

def find_dir_with_most(suffixes, root: Path):
    """Находит папку с максимальным количеством файлов нужных расширений."""
    suffixes = {s.lower() for s in suffixes}
    best = (None, -1)
    for d in root.rglob("*"):
        if not d.is_dir() or is_junk_path(d):
            continue
        n = sum(
            1 for f in d.iterdir()
            if f.is_file() and f.suffix.lower() in suffixes and not is_junk_path(f)
        )
        if n > best[1]:
            best = (d, n)
    return best[0]

# --- Поиск train/test/labels/sample_submission ---
label_dirs = []
for d in DATA_DIR.rglob("*"):
    if d.is_dir() and d.name == "labels" and not is_junk_path(d):
        if any(p.suffix.lower() == ".txt" for p in d.iterdir() if p.is_file()):
            label_dirs.append(d)
assert label_dirs, "Не найдена папка labels с train-разметкой внутри архива."
TRAIN_LABELS_DIR = max(label_dirs, key=lambda d: len(list_labels(d)))

# train images — sibling папки labels
TRAIN_IMAGES_DIR = TRAIN_LABELS_DIR.parent / "images"
assert TRAIN_IMAGES_DIR.exists(), f"Ожидалась train images рядом с labels: {TRAIN_IMAGES_DIR}"

# test images — папка images БЕЗ соседних labels
test_image_dirs = []
for d in DATA_DIR.rglob("*"):
    if d.is_dir() and d.name == "images" and not is_junk_path(d) and d != TRAIN_IMAGES_DIR:
        if not (d.parent / "labels").exists():
            if list_images(d):
                test_image_dirs.append(d)
assert test_image_dirs, "Не найдена папка test/images."
TEST_IMAGES_DIR = max(test_image_dirs, key=lambda d: len(list_images(d)))

# sample_submission.csv
sample_subs = list(DATA_DIR.rglob("sample_submission.csv"))
sample_subs = [p for p in sample_subs if not is_junk_path(p)]
assert sample_subs, "Не найден sample_submission.csv"
SAMPLE_SUBMISSION_PATH = sample_subs[0]

print("TRAIN_IMAGES_DIR :", TRAIN_IMAGES_DIR)
print("TRAIN_LABELS_DIR :", TRAIN_LABELS_DIR)
print("TEST_IMAGES_DIR  :", TEST_IMAGES_DIR)
print("SAMPLE_SUBMISSION:", SAMPLE_SUBMISSION_PATH)
print()
print("train images:", len(list_images(TRAIN_IMAGES_DIR)))
print("train labels:", len(list_labels(TRAIN_LABELS_DIR)))
print("test images :", len(list_images(TEST_IMAGES_DIR)))

## 3. Анализ YOLO-разметки

Разметка в формате YOLO: `class_id xc yc w h` (нормированные координаты в `[0, 1]`). 
Проверяем, что:
- у каждого изображения есть label-файл,
- все строки валидны (5 чисел),
- координаты в диапазоне `[0, 1]`,
- классы идут подряд от 0 до `NC - 1`.

In [ ]:
train_images = list_images(TRAIN_IMAGES_DIR)
train_labels = list_labels(TRAIN_LABELS_DIR)
label_stem_to_path = {p.stem: p for p in train_labels}

rows = []
bad_label_lines = []
out_of_range_boxes = []

for img_path in train_images:
    label_path = label_stem_to_path.get(img_path.stem)
    if label_path is None:
        rows.append({
            "image_path": str(img_path), "label_path": None,
            "image_id": img_path.stem, "n_boxes": 0,
            "classes": [], "has_label": False,
        })
        continue

    classes, n_boxes = [], 0
    with open(label_path, "r") as f:
        for line_idx, line in enumerate(f, start=1):
            parts = line.strip().split()
            if not parts:
                continue
            if len(parts) != 5:
                bad_label_lines.append((str(label_path), line_idx, line.strip()))
                continue
            try:
                cls = int(float(parts[0]))
                x, y, w, h = map(float, parts[1:])
            except Exception:
                bad_label_lines.append((str(label_path), line_idx, line.strip()))
                continue
            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 <= w <= 1 and 0 <= h <= 1):
                out_of_range_boxes.append((str(label_path), line_idx, line.strip()))
            classes.append(cls)
            n_boxes += 1

    rows.append({
        "image_path": str(img_path), "label_path": str(label_path),
        "image_id": img_path.stem, "n_boxes": n_boxes,
        "classes": classes, "has_label": True,
    })

df = pd.DataFrame(rows)

all_classes = [c for cls_list in df["classes"] for c in cls_list]
class_counts = Counter(all_classes)
total_boxes = sum(class_counts.values())

print(f"Train images       : {len(df)}")
print(f"Images без label-а : {int((~df['has_label']).sum())}")
print(f"Битых строк        : {len(bad_label_lines)}")
print(f"Координат вне [0,1]: {len(out_of_range_boxes)}")
print(f"Всего bbox-ов      : {total_boxes}")
print(f"Уникальных классов : {len(class_counts)}")

assert total_boxes > 0
assert len(bad_label_lines) == 0, bad_label_lines[:5]
assert len(out_of_range_boxes) == 0, out_of_range_boxes[:5]

min_class, max_class = min(class_counts), max(class_counts)
assert min_class == 0
assert max_class + 1 == len(class_counts), (
    f"Классы не подряд: min={min_class}, max={max_class}, unique={sorted(class_counts)}"
)

NC = max_class + 1
print(f"\nКоличество классов NC = {NC}")
print("Распределение классов (top-15):",
      dict(sorted(class_counts.items(), key=lambda x: -x[1])[:15]))
df[["image_id", "n_boxes"]].describe().T

In [ ]:
# Sanity check — реально ли читаются картинки и какие у них размеры
sample_check = df.sample(n=min(50, len(df)), random_state=SEED)
shapes, bad_imgs = [], []

for img_path in sample_check["image_path"]:
    img = cv2.imread(img_path)
    if img is None:
        bad_imgs.append(img_path)
    else:
        shapes.append(img.shape[:2])

assert not bad_imgs, bad_imgs[:5]
shape_df = pd.DataFrame(shapes, columns=["height", "width"])
print("Размеры train-картинок (на выборке 50 шт):")
shape_df.describe()

## 4. Train/val split

Стратификация по «главному» классу изображения (первый bbox). Редкие классы (с 1 
примером) сваливаем в один корзинный класс `-1`, чтобы `train_test_split` не падал.

In [ ]:
from sklearn.model_selection import train_test_split

VAL_SIZE = 0.10

df_valid = df[df["has_label"] & df["label_path"].notna()].copy()
df_valid["main_class"] = df_valid["classes"].apply(lambda x: x[0] if x else -1)

freq = df_valid["main_class"].value_counts()
rare = set(freq[freq < 2].index)
df_valid["stratify_class"] = df_valid["main_class"].where(
    ~df_valid["main_class"].isin(rare), -1
)

try:
    train_idx, val_idx = train_test_split(
        df_valid.index, test_size=VAL_SIZE, random_state=SEED,
        shuffle=True, stratify=df_valid["stratify_class"],
    )
    split_type = "stratified"
except ValueError as e:
    print("Stratified split не сработал, fallback на random:", e)
    train_idx, val_idx = train_test_split(
        df_valid.index, test_size=VAL_SIZE, random_state=SEED, shuffle=True,
    )
    split_type = "random"

split_df = df_valid.copy()
split_df["split"] = "train"
split_df.loc[val_idx, "split"] = "val"

print(f"Тип сплита: {split_type}")
print(split_df["split"].value_counts())

train_classes, val_classes = Counter(), Counter()
for _, r in split_df.iterrows():
    (train_classes if r["split"] == "train" else val_classes).update(r["classes"])
missing_in_val = sorted(set(train_classes) - set(val_classes))
print(f"Классов в train: {len(train_classes)}, в val: {len(val_classes)}, "
      f"отсутствует в val: {len(missing_in_val)}")

## 5. Сборка YOLO-датасета

Ultralytics ожидает структуру `images/{train,val}` + `labels/{train,val}` + `data.yaml`.
Кладём всё в `./workdir/yolo_dataset`.

In [ ]:
YOLO_ROOT = WORKDIR / "yolo_dataset"
if YOLO_ROOT.exists():
    shutil.rmtree(YOLO_ROOT)
for split in ["train", "val"]:
    (YOLO_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

for _, row in split_df.iterrows():
    img_path = Path(row["image_path"])
    lbl_path = Path(row["label_path"])
    split = row["split"]
    shutil.copy2(img_path, YOLO_ROOT / "images" / split / img_path.name)
    shutil.copy2(lbl_path, YOLO_ROOT / "labels" / split / lbl_path.name)

data_yaml = {
    "path": str(YOLO_ROOT),
    "train": "images/train",
    "val": "images/val",
    "nc": NC,
    "names": {i: f"class_{i}" for i in range(NC)},
}
DATA_YAML_PATH = YOLO_ROOT / "data.yaml"
with open(DATA_YAML_PATH, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

for split in ["train", "val"]:
    n_img = len(list((YOLO_ROOT / "images" / split).glob("*")))
    n_lbl = len(list((YOLO_ROOT / "labels" / split).glob("*.txt")))
    print(f"{split}: images={n_img}, labels={n_lbl}")
    assert n_img == n_lbl > 0

print("\ndata.yaml:", DATA_YAML_PATH)
print(yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True))

## 6. Обучение YOLO11l

**Архитектура и гиперпараметры — такие же, как в финальном эксперименте:**
- `yolo11l` (~26M параметров) — оптимум для ~1.7K картинок;
- `imgsz=768` важно для мелких объектов (значимая доля bbox-ов <1% площади);
- `AdamW`, `lr0=0.0008`, `cos_lr=True` — стабильнее SGD на этой архитектуре;
- `degrees=0, shear=0, perspective=0` — повороты/искажения ломают bbox-разметку;
- `mosaic=1.0`, `close_mosaic=15` — сильная мозаика, но последние 15 эпох без неё;
- `patience=25`, `epochs=80` — ранняя остановка по плато на val.

Pretrained-веса `yolo11l.pt` ultralytics скачивает из своих GitHub-релизов 
при первом обращении и кэширует в `~/.config/Ultralytics`. В репозитории веса не лежат.

In [ ]:
from ultralytics import YOLO

seed_everything(SEED)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

RUNS_DIR = WORKDIR / "runs"
RUN_NAME = f"yolo11l_seed{SEED}_img768_e80"

# Pretrained-чекпоинт скачивается ultralytics-ом из интернета на лету,
# затем сразу же дообучается под нашу задачу.
model = YOLO("yolo11l.pt")

train_result = model.train(
    data=str(DATA_YAML_PATH),
    epochs=80,
    imgsz=768,
    batch=4,                 # для GPU ~22GB. На меньшей карте — снизить до 2.
    device=DEVICE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    seed=SEED,
    pretrained=True,

    # Оптимизатор
    optimizer="AdamW",
    lr0=0.0008,
    lrf=0.01,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    momentum=0.937,

    # Аугментации (без поворотов/искажений)
    hsv_h=0.015, hsv_s=0.60, hsv_v=0.40,
    degrees=0.0, translate=0.08, scale=0.45,
    shear=0.0, perspective=0.0,
    flipud=0.0, fliplr=0.5,
    mosaic=1.0, mixup=0.08, copy_paste=0.0,
    close_mosaic=15,

    # Контроль
    patience=25,
    save=True, save_period=10,
    cache=False, amp=True,
    workers=2,
    deterministic=True,
    verbose=True, plots=True, val=True,
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_WEIGHTS = Path(model.trainer.best)
LAST_WEIGHTS = Path(model.trainer.last)

print("\nTRAIN FINISHED")
print("  RUN_DIR:", RUN_DIR)
print("  BEST   :", BEST_WEIGHTS)

results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    res = pd.read_csv(results_csv)
    print("\nПоследние 5 эпох:")
    print(res.tail(5).to_string(index=False))
    if "metrics/mAP50(B)" in res.columns:
        print(f"\nBest val mAP50    : {res['metrics/mAP50(B)'].max():.4f}")
        print(f"Best val mAP50-95 : {res['metrics/mAP50-95(B)'].max():.4f}")

## 7. Инференс на test

Финальная конфигурация (выбрана по итогам разведки на валидации в отдельных черновых 
ноутбуках, см. правила сдачи):

- `imgsz=640`, `iou=0.6`, `augment=True` (TTA — flip + multi-scale внутри ultralytics),
- `half=True` — fp16 на GPU,
- порог `conf=0.025` — даёт лучший trade-off precision/recall на val,
- `max_det=80` финальных боксов на картинку.

Порядок строк в `submission.csv` строго совпадает с `sample_submission.csv`.

In [ ]:
from ultralytics import YOLO
from tqdm.auto import tqdm

# Параметры финального инференса
INFER_IMGSZ = 640
INFER_IOU = 0.6
INFER_TTA = True
INFER_BASE_CONF = 0.001  # низкий порог при предсказании, фильтрация — отдельным шагом
MAX_DET_PER_IMAGE = 100
MAX_DET_FINAL = 80
FINAL_CONF_THR = 0.025

# Сопоставляем image_id из sample_submission с реальными файлами test
sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)
test_files = list_images(TEST_IMAGES_DIR)
stem_to_path = {p.stem: p for p in test_files}

missing = [x for x in sample_sub["image_id"].tolist() if x not in stem_to_path]
assert not missing, f"В test/images отсутствуют файлы для {len(missing)} image_id, например: {missing[:3]}"

test_paths = [stem_to_path[x] for x in sample_sub["image_id"].tolist()]
print(f"Test images для предсказания: {len(test_paths)}")

# Загружаем дообученные веса
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
infer_model = YOLO(str(BEST_WEIGHTS))

pred_cache = {}
t0 = time.time()
for img_path in tqdm(test_paths, desc="Inference (TTA)"):
    with Image.open(img_path) as im:
        W, H = im.size
    result = infer_model.predict(
        source=str(img_path),
        imgsz=INFER_IMGSZ,
        conf=INFER_BASE_CONF,
        iou=INFER_IOU,
        max_det=MAX_DET_PER_IMAGE,
        device=DEVICE,
        verbose=False,
        save=False,
        augment=INFER_TTA,
        half=torch.cuda.is_available(),
    )[0]

    preds = []
    if result.boxes is not None and len(result.boxes) > 0:
        xyxy = result.boxes.xyxy.cpu().numpy()
        cls = result.boxes.cls.cpu().numpy().astype(int)
        conf = result.boxes.conf.cpu().numpy()
        for box, c, s in zip(xyxy, cls, conf):
            x1, y1, x2, y2 = box.tolist()
            x1 = max(0, min(int(round(x1)), W - 1))
            y1 = max(0, min(int(round(y1)), H - 1))
            x2 = max(0, min(int(round(x2)), W - 1))
            y2 = max(0, min(int(round(y2)), H - 1))
            if x2 <= x1 or y2 <= y1:
                continue
            preds.append({
                "class_id": int(c), "conf": float(s),
                "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            })
    pred_cache[img_path.stem] = preds
    del result

elapsed = time.time() - t0
avg_dets = np.mean([len(v) for v in pred_cache.values()])
print(f"\nГотово за {elapsed:.1f}s ({elapsed / max(1, len(test_paths)):.2f} s/img). "
      f"Среднее детекций (до фильтра): {avg_dets:.1f}")

## 8. Формирование submission.csv

Формат строки: `class conf x1 y1 x2 y2 [class conf x1 y1 x2 y2 ...]` (по 6 токенов на bbox). 
Если по картинке вообще нет детекций — пишем заглушку `0 0.001 0 0 1 1`, 
чтобы не было пустых ячеек и не сломался парсер.

In [ ]:
PLACEHOLDER = "0 0.001 0 0 1 1"
SUBMISSION_PATH = REPO_ROOT / "submission.csv"

rows = []
n_normal = n_fb_low = n_fb_empty = 0

for image_id in sample_sub["image_id"].tolist():
    preds = pred_cache.get(image_id, [])
    selected = [p for p in preds if p["conf"] >= FINAL_CONF_THR]
    selected = sorted(selected, key=lambda r: r["conf"], reverse=True)[:MAX_DET_FINAL]

    if selected:
        n_normal += 1
        toks = []
        for p in selected:
            toks.extend([
                str(int(p["class_id"])), f"{p['conf']:.6f}",
                str(int(p["x1"])), str(int(p["y1"])),
                str(int(p["x2"])), str(int(p["y2"])),
            ])
        rows.append([image_id, " ".join(toks)])
    elif preds:
        n_fb_low += 1
        best = max(preds, key=lambda r: r["conf"])
        toks = [
            str(int(best["class_id"])), f"{best['conf']:.6f}",
            str(int(best["x1"])), str(int(best["y1"])),
            str(int(best["x2"])), str(int(best["y2"])),
        ]
        rows.append([image_id, " ".join(toks)])
    else:
        n_fb_empty += 1
        rows.append([image_id, PLACEHOLDER])

with open(SUBMISSION_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["image_id", "PredictionString"])
    w.writerows(rows)

# Финальные проверки сабмита
check = pd.read_csv(SUBMISSION_PATH)
assert len(check) == len(sample_sub), (
    f"Размер сабмита ({len(check)}) != размеру sample_submission ({len(sample_sub)})"
)
assert list(check["image_id"]) == list(sample_sub["image_id"]), \
    "Порядок image_id отличается от sample_submission"

check["PredictionString"] = check["PredictionString"].astype(str)
bad = check["PredictionString"].str.split().apply(
    lambda x: (not x) or (len(x) % 6 != 0)
)
assert not bad.any(), f"Битые строки в submission: {bad.sum()}"

avg_boxes = np.mean([len(s.split()) // 6 for s in check["PredictionString"]])
print(f"submission.csv -> {SUBMISSION_PATH}")
print(f"строк: {len(check)} | normal: {n_normal} | fallback_low: {n_fb_low} | empty: {n_fb_empty}")
print(f"среднее bbox/картинку: {avg_boxes:.2f}")
check.head()

---
Готово. Файл `submission.csv` лежит в корне репозитория.